In [ ]:
from datetime import datetime
from pathlib import Path
from typing import Tuple, Dict

from minbpe import RegexTokenizer as Tokenizer
from transformer.model import GPTLanguageModel

import matplotlib.pyplot as plt
import numpy as np
import torch
torch.manual_seed(3647)
torch.set_float32_matmul_precision('high')
from tqdm import tqdm

In [ ]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
data_dir = Path("data") / "ch08"

In [ ]:
tokenizer = Tokenizer()
tokenizer.load(model_file=str(data_dir / "darija_tokenizer.model"))

data = np.load(data_dir / "encoded_atlaset.npy", mmap_mode='r')
print('Data shape:', data.shape)

In [ ]:
parameters = {
  "block_size": 1024,
  "n_embd": 512,
  "n_head": 8,
  "n_layer": 8,
  "dropout": 0.2,
  "vocab_size": len(tokenizer.vocab),
}

# Training Hyperparameters
split_index = int(0.9 * len(data))
batch_size = 4
#learning_rate = 3e-4
min_learning_rate = 3e-5
max_learning_rate = 3e-4
gradient_accumulation_steps = 8
eval_batches = 1_000
#eval_interval = 3000
#save_interval = 10000
eval_interval = 5_000 // gradient_accumulation_steps
#save_interval = 10_000 // gradient_accumulation_steps

# Calculate the number of complete non-overlapping blocks
non_overlapping_blocks = (split_index - 1) // block_size
total_batches_per_epoch = non_overlapping_blocks // batch_size

# Scheduler calculation
max_iters = total_batches_per_epoch * num_epochs
optimizer_steps_total = max_iters // gradient_accumulation_steps
warmup_iters = int(0.1 * optimizer_steps_total)

In [ ]:
model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    block_size=parameters['block_size'],
    n_embd=parameters['n_embd'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    device=device,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=max_learning_rate)

scheduler = CosineAnnealingLR(
    optimizer=optimizer,
    T_max=optimizer_steps_total - warmup_iters,
    eta_min=min_learning_rate,
)

print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

In [ ]:
def get_evaluation_indices(split: str, eval_batches: int) -> torch.Tensor:
    start, end = (0, split_index) if split == 'train' else (split_index, len(data))
    num_blocks = (end - start - 1) // block_size
    return torch.randint(0, num_blocks, (eval_batches,))

def get_batch_for_loss_estimation(split: str, block_indices: list[int]) -> Tuple[torch.Tensor, torch.Tensor]:
    start = 0 if split == 'train' else split_index
    x_batch, y_batch = [], []

    for i in block_indices:
        block_start = start + (i * block_size)
        x = data[block_start:block_start+block_size]
        y = data[block_start+1:block_start+block_size+1]
        x_batch.append(x)
        y_batch.append(y)

    x_batch = torch.tensor(np.array(x_batch), dtype=torch.long).to(device)
    y_batch = torch.tensor(np.array(y_batch), dtype=torch.long).to(device)

    return x_batch, y_batch

@torch.no_grad()
def estimate_loss() -> Dict:
    model.eval()
    output = {}

    for split in ['train', 'val']:
        losses = []
        indices = get_evaluation_indices(split, eval_batches)
        for idx in indices:
            x, y = get_batch_for_loss_estimation(split, [idx.item()])
            _, loss = model(x, y)
            losses.append(loss.item())

        output[split] = sum(losses) / len(losses)

    model.train()
    return output

def save_checkpoint(
    model: GPTLanguageModel,
    optimizer: torch.optim.Optimizer,
    meta: dict,
    prefix: str,
) -> None:
    checkpoint = {
        'meta': meta,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }

    with open(prefix + ".json", 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    torch.save(checkpoint, prefix + ".pt")
    print(f'Saved checkpoint: {prefix + ".pt"}')

In [ ]:
batches_processed = 0
optimizer_steps = 0
train_losses, val_losses = [], []

print(f"Total non-overlapping blocks: {non_overlapping_blocks}")
print(f"Batches per epoch: {total_batches_per_epoch}")
print(f"Total optimizer steps (max_iters): {max_iters}")
print(f"Warmup iterations: {warmup_iters}")

def update_lr(lr):
    for param_group in optimizer.param_groups:
        param_group['lr'] = learning_rate

def optimize(pbar, epoch):
    optimizer_steps += 1

    if optimizer_steps < warmup_iters:
        learning_rate = max_learning_rate * optimizer_steps / warmup_iters
        update_lr(learning_rate)
    elif optimizer_steps == warmup_iters:
        update_lr(max_learning_rate)

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    if optimizer_steps >= warmup_iters:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    pbar.set_postfix(
        loss=f"{loss_val:.4f}",
        lr=f"{current_lr:.6f}",
        step=f"{optimizer_steps}/{optimizer_steps_total}",
    )

    if optimizer_steps % eval_interval == 0:
        eval_losses = estimate_loss()

        print(
            f"{now()} step={optimizer_steps}, "
            f"train_loss={eval_losses['train']:.3f}, "
            f"val loss {eval_losses['val']:.3f}, "
            f"lr={current_lr:.6f}"
        )

        train_losses.append(eval_losses['train'])
        val_losses.append(eval_losses['val'])

        meta = {
            "created_at": now(),
            "parameters": parameters,
            "epoch": epoch,
            "learning_rate": current_lr.
            "train_loss": float(eval_losses['train']),
            "val_loss": float(eval_losses['val']),
        }

        save_checkpoint(
            model=model,
            optimizer=optimizer,
            meta=meta,
            prefix=str(data_dir / "checkpoint_step_{optimizer_steps:07_}",
        )

In [ ]:
model.train()
optimizer.zero_grad(set_to_none=True)

# Training loop
for epoch in range(1, num_epochs+1):
    print(f"Starting epoch {epoch}/{num_epochs}")

    pbar = tqdm(
        iterable=range(0, non_overlapping_blocks, batch_size),
        desc=f"Epoch {epoch+1}/{num_epochs}",
        total=total_batches_per_epoch,
    )

    for i in pbar:
        x_batch, y_batch = [], []
        batch_end_index = min(i + batch_size, non_overlapping_blocks)

        for block_idx in range(i, batch_end_index):
            block_start = block_idx * block_size
            x = data[block_start: block_start + block_size]
            y = data[block_start + 1: block_start + block_size + 1]
            x_batch.append(x)
            y_batch.append(y)

        if not x_batch:
            continue

        x_batch = torch.tensor(np.array(x_batch), dtype=torch.long).to(device)
        y_batch = torch.tensor(np.array(y_batch), dtype=torch.long).to(device)

        # Forward pass
        logits, loss = model(x_batch, y_batch)
        #loss_val = loss.item()
        loss /= gradient_accumulation_steps
        loss.backward()

        batches_processed += 1
        if batches_processed % gradient_accumulation_steps == 0:
            optimize(pbar, epoch)

In [ ]:
if batches_processed % gradient_accumulation_steps != 0:
    optimizer_steps += 1
    if optimizer_steps < warmup_iters:
        learning_rate = max_learning_rate * optimizer_steps / warmup_iters
        update_lr(learning_rate)

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    print(f"\nPerformed final optimizer step ({optimizer_steps}).")

    if optimizer_steps >= warmup_iters:
        scheduler.step()

eval_losses = estimate_loss()
current_lr = optimizer.param_groups[0]['lr']
print(f"{now()} Saving final checkpoint...")

meta = {
    "created_at": now(),
    "parameters": parameters,
    "epoch": epoch,
    "learning_rate": current_lr.
    "train_loss": float(eval_losses['train']),
    "val_loss": float(eval_losses['val']),
}

save_checkpoint(
    model=model,
    optimizer=optimizer,
    meta=meta,
    prefix=str(data_dir / "checkpoint_step_{optimizer_steps:07_}",
)

print("Training finished.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Step")
plt.ylim(0)
plt.ylabel("Loss")
plt.title("Training and Validation Loss Over Time")
plt.legend()
plt.grid()
plt.show()

In [ ]:
input_tokens = tokenizer.encode("ماهي الأسباب الرئيسية اللي خلاو الفرنسيين")

input_tokens = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=100)

print(tokenizer.decode(output[0].tolist()))